In [32]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_classic.retrievers import EnsembleRetriever

In [33]:
doc = [Document(page_content="Langchain is a framework for developing applications powered by language models.", metadata={"source": "langchain"}),
       Document(page_content="Langgraph is a knowledge graph for language models.", metadata={"source": "langgraph"}),
       Document(page_content="Langchain and Langgraph are both tools for working with language models.", metadata={"source": "both"}),
       Document(page_content="The quick brown fox jumps over the lazy dog.", metadata={"source": "foxdog"}),
       Document(page_content="The lazy dog is sleeping.", metadata={"source": "dog"})
       ]

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=doc, embedding=embedding_model)
bm25_retriever = BM25Retriever.from_documents(documents=doc)
retriever = EnsembleRetriever(retrievers=[vectorstore.as_retriever(), bm25_retriever], weights=[0.7, 0.3])

query = "What is Langchain?"
results = retriever.invoke(query)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [34]:
results

[Document(metadata={'source': 'langchain'}, page_content='Langchain is a framework for developing applications powered by language models.'),
 Document(metadata={'source': 'both'}, page_content='Langchain and Langgraph are both tools for working with language models.'),
 Document(metadata={'source': 'dog'}, page_content='The lazy dog is sleeping.'),
 Document(metadata={'source': 'langgraph'}, page_content='Langgraph is a knowledge graph for language models.')]

In [35]:
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

In [49]:
from dotenv import load_dotenv
import os
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
prompt = PromptTemplate.from_template("""Answer the question based on the following documents:
                                      {context}
                                      Question: {input}
                                      Answer:""")

llm = init_chat_model(model_provider="groq",model="llama-3.1-8b-instant")

llm.invoke("Hello, world!")

AIMessage(content="Hello, world. It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 39, 'total_tokens': 66, 'completion_time': 0.028655909, 'completion_tokens_details': None, 'prompt_time': 0.001858891, 'prompt_tokens_details': None, 'queue_time': 0.046397989, 'total_time': 0.0305148}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4330-cb74-79c3-bc5e-3f86c7215034-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 39, 'output_tokens': 27, 'total_tokens': 66})

In [50]:
doc_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
retrieval_chain = create_retrieval_chain(retriever, doc_chain)
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001BBAD2C4E10>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001BBACF5EAD0>)], weights=[0.7, 0.3]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='Answer the question based on the following documents:\n                                      {context}\n                            

In [52]:
retrieval_chain.invoke({"input": "What is Langchain?"})

{'input': 'What is Langchain?',
 'context': [Document(metadata={'source': 'langchain'}, page_content='Langchain is a framework for developing applications powered by language models.'),
  Document(metadata={'source': 'both'}, page_content='Langchain and Langgraph are both tools for working with language models.'),
  Document(metadata={'source': 'dog'}, page_content='The lazy dog is sleeping.'),
  Document(metadata={'source': 'langgraph'}, page_content='Langgraph is a knowledge graph for language models.')],
 'answer': 'Langchain is a framework for developing applications powered by language models.'}